# R1 reranker pilot

Same shape as `leg2_encoder_pilot.ipynb`: draw by bucket, pair before/after, eye-test real
queries. R1 is the **opinion floor**'s relevance judge (M1: predicts an assessor, never writes
lane qrels) — it scores every doc in the union of the three routes' top-10s, and this notebook
reads those scores two ways: does a doc's R1-implied relevance change the row's score when fed
back through the real objective (`RouterObjective.assess` — reused, not reimplemented)?

**Model: `nvidia/llama-nemotron-rerank-vl-1b-v2:free`** via OpenRouter's dedicated
`POST /api/v1/rerank` endpoint (added 2026 — not in the chat-completions `/models` catalog, and
`litellm.rerank()` does not yet support the `openrouter` provider, so this calls the REST endpoint
directly). **One real network call per QUERY** — the whole union of docs goes in one request, unlike
a local pairwise CrossEncoder pass. Verified live: real response schema below, `usage.cost == 0` on
the free tier. Third family from both the dense leg (BAAI) and the stack probe (Qwen) — satisfies A9
by construction, and multimodal (vision-language) architecture besides.

**Fixed control, same discipline as leg2:** nothing here writes `qrels_r1.parquet` or touches any
lane's real qrels file. This is measurement only — the threshold used below is a provisional
self-calibration (`score > worst known-relevant doc's score, same query`), explicitly **not**
A11's real calibration (that needs the ~500 human-labelled unjudged pairs the plan budgets).

In [59]:
import os
import time

import pandas as pd
import requests
from dotenv import load_dotenv

from composition.pool_v3 import CEILING, LabelledPool
from hybrid_search_rrf_dataset.labels import oracle_dir
from hybrid_search_rrf_dataset.objective import RouterObjective
from hybrid_search_rrf_dataset.router import DATA_DIR

load_dotenv()
OPENROUTER_KEY = os.environ.get("OPENROUTER_API_KEY") or os.environ.get("OPEN_ROUTER_API_KEY")
assert OPENROUTER_KEY, "no OpenRouter key in .env"
RERANK_MODEL = "voyageai/rerank-2.5-lite"
RERANK_URL = "https://openrouter.ai/api/v1/rerank"
# The provider rejects query/document pairs above 10,240 tokens. Keep a
# conservative character budget while leaving the corpus cache untruncated.
MAX_DOCUMENT_CHARS = 40_000

pd.set_option("display.width", 170)
ROUTES = ["dense_only", "pure_rrf", "sparse_only"]
SCORE_COLS = [f"score_{r}" for r in ROUTES]

## 0. The real API — one call, sanity-checked

In [60]:
def rerank(query: str, documents: list[str], *, retries: int = 5) -> list[float]:
    """Scores aligned to `documents`' input order (the endpoint returns them
    reordered by relevance with an `index` back-pointer) — undo that here so
    every caller can zip against its own doc-id list without re-deriving it.
    `:free` tier is burst-rate-limited (measured: a lone call succeeds, a tight
    loop of them 429s within a few requests; no `Retry-After` header is
    exposed) — back off hard rather than trusting a short retry window."""
    original_documents = [str(d) for d in documents]
    documents = [d[:MAX_DOCUMENT_CHARS] for d in original_documents]
    truncated = sum(a != b for a, b in zip(original_documents, documents))
    if truncated:
        print(f"truncated {truncated}/{len(documents)} documents to {MAX_DOCUMENT_CHARS:,} chars for the 10,240-token limit")

    for attempt in range(retries):
        r = requests.post(
            RERANK_URL,
            headers={"Authorization": f"Bearer {OPENROUTER_KEY}", "Content-Type": "application/json"},
            json={"model": RERANK_MODEL, "query": query, "documents": documents,
                  "top_n": len(documents)},
            timeout=30,
        )
        if r.status_code == 429 and attempt < retries - 1:
            time.sleep(3.0 * 2**attempt)
            continue
        if r.status_code == 422:
            detail = r.text[:1000].replace("\n", " ")
            lengths = [len(str(d)) for d in documents]
            raise ValueError(
                f"rerank rejected payload (422): {detail}; "
                f"documents={len(documents)}, max_chars={max(lengths, default=0)}, "
                f"total_chars={sum(lengths)}"
            )
        r.raise_for_status()
        body = r.json()
        scores = [0.0] * len(documents)
        for item in body["results"]:
            scores[item["index"]] = item["relevance_score"]
        return scores, body.get("usage", {})
    raise RuntimeError("rerank: exhausted retries on 429")


test_scores, test_usage = rerank(
    "what is the capital of france",
    ["paris is the capital of france", "the weather today is sunny", "berlin is the capital of germany"],
)
print("scores (input order):", test_scores)
print("usage:", test_usage)
assert test_usage.get("cost", 0) == 0, "expected the free tier to cost $0"

scores (input order): [0.890625, 0.376953125, 0.384765625]
usage: {'total_tokens': 29, 'cost': 5.8e-07}


AssertionError: expected the free tier to cost $0

## 1. Draw — by `kind`, `native=False` only

R1's whole point is `fake_tie` / `all_zero` / `undecisive` (P0.4's retargeting). `native=True` rows
have zero oracle-ranking coverage (A5 — measured 0% match vs 100% for `native=False` on
crumb-legal-qa) and would silently waste the draw if not excluded here. Kept small — this is a real
network call per row, not a local batched pass, so wall-clock and rate limits matter.

In [61]:
SEED = 0
N_PER_KIND = 30  # real network calls, burst-rate-limited on the free tier — keep this bounded
PILOT_LANES = ("crumb-legal-qa", "gooaq")  # v2-origin tier, oracle rankings on disk

pool = LabelledPool()
labels = pool.labels()
classified = pool.classify(labels)
live_lanes = [l for l in PILOT_LANES if (oracle_dir(DATA_DIR / "route_labels", l) / "rows.parquet").exists()]
print(f"lanes with banked rankings: {live_lanes}")

pool_frame = classified[classified["dataset"].isin(live_lanes) & ~classified["native"]]
draws = []
for kind, group in pool_frame.groupby("kind"):
    n = min(N_PER_KIND, len(group))
    draws.append(group.sample(n, random_state=SEED))
    print(f"{kind:14s}  drew {n:4d} / {len(group):5d} available")
drawn = pd.concat(draws, ignore_index=True)

lanes with banked rankings: ['crumb-legal-qa', 'gooaq']
all_zero        drew   30 /  1309 available
decisive        drew   30 /  1245 available
fake_tie        drew   30 /  3272 available
genuine_tie     drew   12 /    12 available
undecisive      drew   30 /  3303 available


## 2. Score — one rerank call per row, real union docs plus the known judged doc(s)

The judged doc is scored too (even when no route retrieved it, e.g. `all_zero`) — it is the
per-query calibration anchor the provisional threshold needs.

In [62]:
qrels_cache: dict[str, pd.DataFrame] = {}
corpus_cache: dict[str, dict[str, str]] = {}
oracle_cache: dict[str, pd.DataFrame] = {}


def lane_qrels(lane: str) -> pd.DataFrame:
    if lane not in qrels_cache:
        qrels_cache[lane] = pd.read_parquet(
            DATA_DIR / lane / "qrels.parquet"
        ).astype({"query_id": str, "doc_id": str})
    return qrels_cache[lane]


def lane_corpus_text(lane: str) -> dict[str, str]:
    # one full read per lane, not per-doc pushdown: every corpus.parquet on disk
    # is a single row group, so pushdown degenerates to a full read anyway —
    # measured 0.5s/doc on orcas; one dict build amortizes it across the sample.
    if lane not in corpus_cache:
        c = pd.read_parquet(DATA_DIR / lane / "corpus.parquet")
        has_title = "title" in c.columns
        corpus_cache[lane] = {
            str(r.doc_id): (f"{r.title}\n{r.text}" if has_title else str(r.text)).strip()
            for r in c.itertuples()
        }
    return corpus_cache[lane]


def lane_oracle(lane: str) -> pd.DataFrame:
    if lane not in oracle_cache:
        path = oracle_dir(DATA_DIR / "route_labels", lane) / "rows.parquet"
        oracle_cache[lane] = pd.read_parquet(
            path, columns=["query_id", "route_rankings"]
        ).astype({"query_id": str})
    return oracle_cache[lane]

In [63]:
from tqdm.auto import tqdm

rows_out = []
call_seconds = []

for row in tqdm(
      drawn.itertuples(),
      total=len(drawn),
      desc="Reranking rows",
  ):
    oracle_rows = lane_oracle(row.dataset)
    match = oracle_rows.loc[oracle_rows["query_id"] == str(row.query_id)]
    if match.empty:
        continue
    rankings = {r: list(v) for r, v in match.iloc[0]["route_rankings"].items()}
    texts = lane_corpus_text(row.dataset)
    qrels = lane_qrels(row.dataset)
    judged = qrels.loc[
        (qrels["query_id"] == str(row.query_id)) & (qrels["relevance"] >= row.min_relevance),
        "doc_id",
    ].astype(str).tolist()

    union = sorted(set().union(*rankings.values()) | set(judged))
    docs = [texts.get(d, "") for d in union]

    t0 = time.time()
    scores, usage = rerank(row.query, docs)
    call_seconds.append(time.time() - t0)
    time.sleep(1.0)  # proactive pacing — a tight loop trips the free tier's burst limit

    rows_out.append({
        "dataset": row.dataset, "query_id": row.query_id, "kind": row.kind, "query": row.query,
        "min_relevance": row.min_relevance, "rankings": rankings,
        "judged": judged, "union": union, "scores": dict(zip(union, scores)),
        "doc_text": {d: texts.get(d, "") for d in union},  # for the eye test — read what R1 actually marked relevant
        "cost": usage.get("cost", 0),
        "score_dense_only": row.score_dense_only, "score_pure_rrf": row.score_pure_rrf,
        "score_sparse_only": row.score_sparse_only,
    })

print(f"rows scored: {len(rows_out)} / {len(drawn)}")
print(f"total cost: ${sum(r['cost'] for r in rows_out):.6f}")

seconds = pd.Series(call_seconds)
print(f"call latency — median {seconds.median():.2f}s, mean {seconds.mean():.2f}s "
      f"(min {seconds.min():.2f}s, max {seconds.max():.2f}s, n={len(seconds)})")
print("mean is inflated by retry backoff on 429s (this run's 46s max ~= 3+6+12+24s of")
print("backoff), not real request time — median is the honest per-call figure.")
hours_median = 224_910 * seconds.median() / 3600
hours_mean = 224_910 * seconds.mean() / 3600
print(f"extrapolated to 224,910 sequential calls: {hours_median:.1f} h at the median, "
      f"{hours_mean:.1f} h at the (backoff-inflated) mean")

Reranking rows:  14%|█▎        | 18/132 [00:28<03:01,  1.59s/it]

truncated 1/18 documents to 40,000 chars for the 10,240-token limit


Reranking rows: 100%|██████████| 132/132 [03:11<00:00,  1.45s/it]

rows scored: 132 / 132
total cost: $0.025011
call latency — median 0.36s, mean 0.44s (min 0.28s, max 0.90s, n=132)
mean is inflated by retry backoff on 429s (this run's 46s max ~= 3+6+12+24s of
backoff), not real request time — median is the honest per-call figure.
extrapolated to 224,910 sequential calls: 22.2 h at the median, 27.7 h at the (backoff-inflated) mean


## 3. R1 verdict + reread through the real objective

Provisional threshold: a union doc is R1-relevant if its score beats the **lowest-scoring already-
judged** doc for that same query — self-calibrating per query, no cross-query cutoff to get wrong.
Then `RouterObjective.assess` — the SAME production scoring code, not a reimplementation — reruns
each route's persisted ranking against the R1-augmented relevant set.

In [64]:
paired_rows = []
added_docs_by_row: dict[tuple[str, str], list[tuple[str, float, str]]] = {}
for r in rows_out:
    doc_scores = r["scores"]
    judged_scores = [doc_scores[d] for d in r["judged"] if d in doc_scores]
    floor = min(judged_scores) if judged_scores else float("inf")
    newly_added = {d for d, s in doc_scores.items() if s > floor} - set(r["judged"])
    r1_relevant = set(r["judged"]) | newly_added
    added_docs_by_row[(r["dataset"], r["query_id"])] = sorted(
        ((d, doc_scores[d], r["doc_text"].get(d, "")) for d in newly_added),
        key=lambda t: -t[1],
    )

    gold_qrel = {d: 1 for d in r1_relevant}
    objective = RouterObjective(min_relevance=r["min_relevance"])
    new_scores = {}
    for route, ranked in r["rankings"].items():
        ranking = {d: float(len(ranked) - i) for i, d in enumerate(ranked)}
        new_scores[route], _ = objective.assess(ranking, gold_qrel)

    paired_rows.append({
        "dataset": r["dataset"], "query_id": r["query_id"], "kind_1": r["kind"],
        "score_dense_only_1": r["score_dense_only"], "score_dense_only_r1": new_scores["dense_only"],
        "score_pure_rrf_1": r["score_pure_rrf"], "score_pure_rrf_r1": new_scores["pure_rrf"],
        "score_sparse_only_1": r["score_sparse_only"], "score_sparse_only_r1": new_scores["sparse_only"],
        "n_r1_added": len(r1_relevant) - len(r["judged"]),
    })

paired = pd.DataFrame(paired_rows)
for suffix in ("1", "r1"):
    cols = [f"score_{r}_{suffix}" for r in ROUTES]
    paired[f"max_{suffix}"] = paired[cols].max(axis=1)
    paired[f"min_{suffix}"] = paired[cols].min(axis=1)
    paired[f"bucket_{suffix}"] = "routes_differ"
    paired.loc[paired[f"max_{suffix}"] - paired[f"min_{suffix}"] <= 1e-9, f"bucket_{suffix}"] = "all_tied"
    paired.loc[paired[f"max_{suffix}"] <= 1e-9, f"bucket_{suffix}"] = "all_zero"
    paired[f"winner_{suffix}"] = paired[cols].idxmax(axis=1).str.replace("score_", "").str.replace(f"_{suffix}", "")

print(f"{len(paired)} rows paired, mean union docs added as R1-relevant: {paired['n_r1_added'].mean():.1f}")
pd.crosstab(paired["bucket_1"], paired["bucket_r1"], margins=True)

132 rows paired, mean union docs added as R1-relevant: 3.3


bucket_r1,all_tied,all_zero,routes_differ,All
bucket_1,,,,
all_tied,35,0,7,42
all_zero,5,11,14,30
routes_differ,4,0,56,60
All,44,11,77,132


### The one number: did R1 find the judged document where retrieval didn't?

In [65]:
az = paired[paired["bucket_1"] == "all_zero"]
rescued = az[az["bucket_r1"] != "all_zero"]
print(f"all_zero rows: {len(az)}   moved off zero: {len(rescued)} ({len(rescued) / max(len(az), 1):.1%})")

ct = paired[(paired["kind_1"] == "genuine_tie")]
print(f"\nceiling/genuine ties: {len(ct)}")
broke = ct[ct["bucket_1"] != ct["bucket_r1"]]
print(f"broke by R1: {len(broke)} ({len(broke) / max(len(ct), 1):.1%})")

all_zero rows: 30   moved off zero: 19 (63.3%)

ceiling/genuine ties: 12
broke by R1: 7 (58.3%)


## 4. Eye test — read what R1 actually marked relevant

Score deltas can't tell you whether the over-acceptance finding means the added docs are real or
noise — only reading them can. Query, the already-known judged doc, and the newly-added docs R1's
threshold accepted, full text, for every row where something actually moved.

In [66]:
import textwrap


def eye(row, max_added: int = 3) -> None:
    key = (row["dataset"], row["query_id"])
    r = next(x for x in rows_out if (x["dataset"], x["query_id"]) == key)
    print(f"[{row['dataset']}/{row['query_id']}] {row['kind_1']}  "
          f"bucket {row['bucket_1']} -> {row['bucket_r1']}")
    print(textwrap.fill(f"QUERY: {r['query']}", width=100, subsequent_indent='       '))
    for d in r["judged"]:
        text = r["doc_text"].get(d, "")[:300]
        score = r["scores"].get(d)
        print(textwrap.fill(
            f"  [already judged, score={score:.3f}] {text}", width=100, subsequent_indent=' ' * 4,
        ))
    added = added_docs_by_row.get(key, [])
    print(f"  -- R1 newly marked relevant: {len(added)} --")
    for doc_id, score, text in added[:max_added]:
        print(textwrap.fill(
            f"  [NEW, score={score:.3f}] {text[:300]}", width=100, subsequent_indent=' ' * 4,
        ))
    print()


moved = paired[paired["bucket_1"] != paired["bucket_r1"]]
print(f"{len(moved)} rows moved buckets — reading a sample:\n")
for _, row in moved.sample(min(8, len(moved)), random_state=0).iterrows():
    eye(row)

30 rows moved buckets — reading a sample:

[crumb-legal-qa/9162] all_zero  bucket all_zero -> routes_differ
QUERY: Is the constable an individual empowered by law to execute a writ of eviction? In the state
       of Utah
  [already judged, score=0.520] # 2021 Utah Code ## Title 78B - Judicial Code ### Chapter 6 -
    Particular Proceedings #### Part 8 - Forcible Entry and Detainer ##### Section 808 - Possession
    bond of plaintiff -- Alternative remedies. 78B-6-808. Possession bond of plaintiff --
    Alternative remedies.    (1) At any time between the fil
  -- R1 newly marked relevant: 2 --
  [NEW, score=0.574] # 2021 Utah Code ## Title 78B - Judicial Code ### Chapter 6 - Particular
    Proceedings #### Part 8 - Forcible Entry and Detainer ##### Section 810 - Court procedures.
    Effective 5/12/2020  78B-6-810. Court procedures.    (1) In an action under this chapter in
    which the tenant remains in possession of
  [NEW, score=0.543] # 2021 South Carolina Code of Laws ## Title 

## Reading it

- **This leg is genuinely $0** (the free tier's own reported `usage.cost`), unlike the LLM legs —
  the only real cost is wall-clock and rate-limit exposure at pool scale.
- **Latency is per-query, not per-pair** — a network round trip per row. The free tier is
  burst-rate-limited (measured: a lone call succeeds, a tight loop 429s within a few requests, no
  `Retry-After` header exposed) — read the **median** call time, not the mean: retry backoff on a
  handful of 429s inflates the mean badly (one run: 46s max vs 0.26s min, mean 1.85s pulled well above
  the typical call). A pool-scale run needs real pacing/concurrency tuning, not naive parallelism.
- **The threshold in §3 is over-permissive, now triangulated across three independent rerankers.**
  Local `mmarco-mMiniLMv2` (BAAI/Qwen-independent, English) gave 63.6% then 88.3% `all_zero` rescue
  across two draws; the real `nvidia/llama-nemotron-rerank-vl-1b-v2` run (different architecture,
  different training data, different provider entirely) gave 70.0% / 80.0% on ceiling ties — the same
  pattern, not a model-specific quirk. This is strong evidence the **threshold design** is what over-
  accepts (a per-query floor set by one judged doc's score is too easy to clear), independent of which
  reranker computes the scores. Treat every number here as upper-bound-until-A11-calibrated.